# Project Summary: Property Recommender System

This notebook documents the end-to-end process of building a semantic real estate recommendation engine. Below is a summary of the completed steps:

1. **Environment Setup & Data Loading**
   - Installed necessary libraries: `langchain`, `langchain-groq`, `langchain-huggingface`, `chromadb`, and `sentence-transformers`.
   - Loaded property data from `nawy_properties_cleaned.csv` into a pandas DataFrame.

2. **Vector Database Initialization**
   - Unzipped and loaded a pre-built Chroma DB containing property embeddings.
   - Configured `HuggingFaceEmbeddings` using the `Qwen/Qwen3-Embedding-0.6B` model for semantic understanding.

3. **Semantic Search Execution**
   - Performed a similarity search against the vector database using a natural language query (e.g., *'Apartment in New Cairo less than 15 million with 2 bedrooms'*).
   - Retrieved the top 50 most relevant property candidates based on vector similarity.

4. **LLM-Powered Filtering**
   - Integrated **Groq** (using `gpt-oss-20b`) to dynamically generate Python code for precise metadata filtering.
   - Applied the generated code to the candidate list to filter by specific attributes like `price_float` and `Beds` count.

5. **Data Processing & Export**
   - Filtered the original DataFrame to include only the top-matching and validated properties.
   - Exported the final recommendations into JSON format and prepared a clean context string for further consultation or user display.

In [7]:
!unzip property_recommender_db.zip -d ./

Archive:  property_recommender_db.zip
   creating: ./property_recommender_db/
  inflating: ./property_recommender_db/chroma.sqlite3  
   creating: ./property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/
  inflating: ./property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/index_metadata.pickle  
  inflating: ./property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/length.bin  
  inflating: ./property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/data_level0.bin  
  inflating: ./property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/header.bin  
  inflating: ./property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/link_lists.bin  


In [6]:
# !rm -r /content/property_recommender_db

rm: cannot remove '/content/ee6ae8fd-8016-47d1-a235-ebd25259d855': No such file or directory


In [ ]:
%pip install langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers

In [29]:
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
import pandas as pd
import json

In [24]:
df = pd.read_csv("nawy_properties_cleaned.csv", dtype={"id": str})
df.head()

,id,location,property_name,description,m2,Beds,Baths,payment_plan,price,url_path,tag,cover_image,developer_logo,price_float,embedding_text
0,00000,New Capital City,"Apartment, Celia",Apartment for sale in Celia - with 2 bedrooms ...,101.0,2.0,1.0,"32,083 EGP monthly / 2 Years","4,200,000 EGP",https://www.nawy.com/compound/623-celia/proper...,Resale,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,4200000.0,id: 00000 | location: New Capital City | prope...
1,00001,6th settlement,"Apartment, Noi",Apartment for sale in Noi - with 1 bedroom in ...,81.0,1.0,1.0,"274,567 EGP quarterly / 8 Years","8,786,151 EGP",https://www.nawy.com/compound/1364-noi/propert...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,8786151.0,id: 00001 | location: 6th settlement | propert...
2,00002,6th settlement,"Apartment, Noi",Apartment for sale in Noi - with 1 bedroom in ...,73.0,1.0,1.0,"195,914 EGP quarterly / 10 Years","8,707,294 EGP",https://www.nawy.com/compound/1364-noi/propert...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,8707294.0,id: 00002 | location: 6th settlement | propert...
3,00003,El Sheikh Zayed,"Apartment, Hyatt Centric",Apartment for sale in Hyatt Centric - with 1 b...,122.0,1.0,2.0,"351,370 EGP quarterly / 11 Years","16,274,000 EGP",https://www.nawy.com/compound/1643-hyatt-centr...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,16274000.0,id: 00003 | location: El Sheikh Zayed | proper...
4,00004,6th of October City,"Apartment, Green 6",Apartment for sale in Green 6 - with 2 bedroom...,121.0,2.0,2.0,"26,272 EGP quarterly / 5.6 Years","4,588,500 EGP",https://www.nawy.com/compound/635-green-6/prop...,Resale,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,4588500.0,id: 00004 | location: 6th of October City | pr...


In [12]:
# Embeddings - local Qwen model (downloads on first use ~1.19G)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"trust_remote_code": True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

In [13]:
# Load persisted DB (no from_documents needed)
vectorstore = Chroma(
    persist_directory="./property_recommender_db",  # Your exact path
    embedding_function=embeddings
)

In [14]:
print(f"Collection count: {vectorstore._collection.count()}")
print("Sample metadata:", vectorstore.similarity_search("test", k=1)[0].metadata)

Collection count: 8597
Sample metadata: {'source': 'id: 00788 | location: 6th of October City | property_name: Apartment, Nmq | description: Apartment for sale in Nmq - with 3 bedrooms in 6th of October City by Melee Development. | Beds: 3.0 | Baths: 3.0 | m2: 173.0', 'property_name': 'Apartment, Nmq', 'price_float': '18346000.0', 'Baths': '3.0', 'm2': '173.0', 'location': '6th of October City', 'Beds': '3.0', 'id': '00788', 'row': 788}


In [46]:
query = "Apartment in New Cairo less than 15 million with 2 bedromms "
recommendations = vectorstore.similarity_search(
    query,
    k=50
)

recommendation_ids = []
for doc in recommendations:
    recommendation_ids.append(doc.metadata.get('id'))

print("Recommendation IDs:", recommendation_ids)

Recommendation IDs: ['00408', '01091', '00050', '00760', '00357', '00440', '01212', '00556', '00644', '00704', '00922', '00319', '00675', '00694', '01079', '00338', '00687', '01008', '00307', '01022', '00292', '01011', '00329', '00570', '00132', '00552', '01089', '00798', '00207', '00636', '00978', '00906', '00918', '00684', '00261', '00404', '00326', '01017', '00020', '00068', '00019', '00673', '00690', '00885', '01006', '01210', '00018', '00017', '00685', '00299']


In [47]:
recommended_df = df[df['id'].isin(recommendation_ids)]
recommended_df.head()

,id,location,property_name,description,m2,Beds,Baths,payment_plan,price,url_path,tag,cover_image,developer_logo,price_float,embedding_text
17,00017,New Cairo,"Apartment, Address East",Apartment for sale in Address East - with 3 be...,147.0,3.0,2.0,"493,707 EGP quarterly / 4 Years","9,293,299 EGP",https://www.nawy.com/compound/1935-address-eas...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,9293299.0,id: 00017 | location: New Cairo | property_nam...
18,00018,New Cairo,"Apartment, Address East",Apartment for sale in Address East - with 3 be...,167.0,3.0,2.0,"560,878 EGP quarterly / 4 Years","10,557,694 EGP",https://www.nawy.com/compound/1935-address-eas...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,10557694.0,id: 00018 | location: New Cairo | property_nam...
19,00019,New Cairo,"Apartment, Address East",Apartment for sale in Address East - with 3 be...,171.0,3.0,2.0,"508,578 EGP quarterly / 4 Years","9,573,238 EGP",https://www.nawy.com/compound/1935-address-eas...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,9573238.0,id: 00019 | location: New Cairo | property_nam...
20,00020,New Cairo,"Apartment, Address East",Apartment for sale in Address East - with 3 be...,160.0,3.0,2.0,"475,863 EGP quarterly / 4 Years","8,957,416 EGP",https://www.nawy.com/compound/1935-address-eas...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,8957416.0,id: 00020 | location: New Cairo | property_nam...
50,00050,New Cairo,"Apartment, Mist",Apartment for sale in Mist - with 3 bedrooms i...,157.0,3.0,3.0,"360,223 EGP quarterly / 10 Years","16,009,897 EGP",https://www.nawy.com/compound/1528-mist/proper...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,16009897.0,id: 00050 | location: New Cairo | property_nam...


In [31]:
from google.colab import userdata

# Retrieve the secret by the name you gave it
groq_api_key = userdata.get('GROQ_API_KEY')

# Now you can use it in your logic
print("API Key loaded successfully!")

API Key loaded successfully!


In [48]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=5000)

# sample = recommended_df.head(2).to_string()
# Sample rows: {sample}
columns = str(['m2', 'Beds', 'Baths', 'price_float'])
# query = "Apartment in October City more than 10 million with 2 bedrooms"

prompt = ChatPromptTemplate.from_template("""
Given this DataFrame info:
Columns: {columns}

Generate Python pandas code based on just the columns above to this: {query}.
Rules:
- Use ONLY the provided columns.
- Store the result in a variable called df_filtered.
- The output must be a pandas filtering operation on recommended_df.
- Return ONLY the pandas code (no markdown, no explanation).
""")

chain = prompt | llm


code = chain.invoke({
    "columns": columns,
    "query": query
}).content

# Print generated code
print("Generated Code:\n")
print(code)

# Execute generated code
# exec(code)

Generated Code:

df_filtered = recommended_df[(recommended_df['price_float'] < 15000000) & (recommended_df['Beds'] == 2)]


In [49]:
# Execute generated code
exec(code)

In [50]:
df_filtered.head()

,id,location,property_name,description,m2,Beds,Baths,payment_plan,price,url_path,tag,cover_image,developer_logo,price_float,embedding_text
207,00207,South New Cairo,"Apartment, DISTRICT 5",Apartment for sale in DISTRICT 5 - with 2 bedr...,149.0,2.0,3.0,"20,000 EGP monthly / 5 Years","13,200,000 EGP",https://www.nawy.com/compound/222-district-5/p...,Resale,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,13200000.0,id: 00207 | location: South New Cairo | proper...
552,00552,South New Cairo,"Apartment, DISTRICT 5",Apartment for sale in DISTRICT 5 - with 2 bedr...,125.0,2.0,3.0,"36,160 EGP monthly / 6 Years","9,281,000 EGP",https://www.nawy.com/compound/222-district-5/p...,Resale,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,9281000.0,id: 00552 | location: South New Cairo | proper...
556,00556,New Cairo,"Apartment, Trio",Apartment for sale in Trio - with 2 bedrooms ...,130.0,2.0,3.0,NaN,"14,095,338 EGP",https://www.nawy.com/compound/204-trio/propert...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,14095338.0,id: 00556 | location: New Cairo | property_nam...
636,00636,New Cairo,"Apartment, Address East",Apartment for sale in Address East - with 2 be...,121.0,2.0,1.0,"438,746 EGP quarterly / 4 Years","8,258,739 EGP",https://www.nawy.com/compound/1935-address-eas...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,8258739.0,id: 00636 | location: New Cairo | property_nam...
684,00684,New Cairo,"Apartment, Origami",Apartment for sale in Origami - with 2 bedroo...,115.0,2.0,2.0,NaN,"9,862,824 EGP",https://www.nawy.com/compound/1246-origami/pro...,NaN,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,9862824.0,id: 00684 | location: New Cairo | property_nam...


In [51]:
df_filtered.shape

(13, 15)

In [53]:
df_filtered.columns

Index(['id', 'location', 'property_name', 'description', 'm2', 'Beds', 'Baths',
       'payment_plan', 'price', 'url_path', 'tag', 'cover_image',
       'developer_logo', 'price_float', 'embedding_text'],
      dtype='object')

In [52]:
df_filtered_json = df_filtered.to_json(orient='records')
print(df_filtered_json)

[{"id":"00207","location":"South New Cairo","property_name":"Apartment, DISTRICT 5","description":"Apartment for sale in DISTRICT 5 - with 2 bedrooms in South New Cairo by Marakez.","m2":149.0,"Beds":2.0,"Baths":3.0,"payment_plan":"20,000 EGP monthly \/ 5 Years","price":"13,200,000 EGP","url_path":"https:\/\/www.nawy.com\/compound\/222-district-5\/property\/55183-apartment-for-sale-in-district-5-new-cairo","tag":"Resale","cover_image":"https:\/\/prod-images.cooingestate.com\/processed\/property_image\/image\/271096\/high.webp","developer_logo":"https:\/\/prod-images.cooingestate.com\/processed\/developer\/logo_image\/65\/medium.webp","price_float":13200000.0,"embedding_text":"id: 00207 | location: South New Cairo | property_name: Apartment, DISTRICT 5 | description: Apartment for sale in DISTRICT 5 - with 2 bedrooms in South New Cairo by Marakez. | Beds: 2.0 | Baths: 3.0 | m2: 149.0 | tag: Resale"},{"id":"00552","location":"South New Cairo","property_name":"Apartment, DISTRICT 5","desc

In [55]:
# this would be used as an input for llm to work as a consaltant to compare and give me the best options

cols_to_keep = [
    'id', 'location', 'property_name', 'description',
    'm2', 'Beds', 'Baths', 'payment_plan', 'price', 'tag'
]

df_recommendations_input = df_filtered[cols_to_keep]
df_recommendations_input.head()

,id,location,property_name,description,m2,Beds,Baths,payment_plan,price,tag
207,00207,South New Cairo,"Apartment, DISTRICT 5",Apartment for sale in DISTRICT 5 - with 2 bedr...,149.0,2.0,3.0,"20,000 EGP monthly / 5 Years","13,200,000 EGP",Resale
552,00552,South New Cairo,"Apartment, DISTRICT 5",Apartment for sale in DISTRICT 5 - with 2 bedr...,125.0,2.0,3.0,"36,160 EGP monthly / 6 Years","9,281,000 EGP",Resale
556,00556,New Cairo,"Apartment, Trio",Apartment for sale in Trio - with 2 bedrooms ...,130.0,2.0,3.0,NaN,"14,095,338 EGP",NaN
636,00636,New Cairo,"Apartment, Address East",Apartment for sale in Address East - with 2 be...,121.0,2.0,1.0,"438,746 EGP quarterly / 4 Years","8,258,739 EGP",NaN
684,00684,New Cairo,"Apartment, Origami",Apartment for sale in Origami - with 2 bedroo...,115.0,2.0,2.0,NaN,"9,862,824 EGP",NaN


In [59]:
context = df_recommendations_input.to_string(index=False)
print(context)

   id        location                         property_name                                                                                                       description    m2  Beds  Baths                     payment_plan          price    tag
00207 South New Cairo                 Apartment, DISTRICT 5                                 Apartment for sale in DISTRICT 5 - with 2 bedrooms in South New Cairo by Marakez. 149.0   2.0    3.0     20,000 EGP monthly / 5 Years 13,200,000 EGP Resale
00552 South New Cairo                 Apartment, DISTRICT 5                                 Apartment for sale in DISTRICT 5 - with 2 bedrooms in South New Cairo by Marakez. 125.0   2.0    3.0     36,160 EGP monthly / 6 Years  9,281,000 EGP Resale
00556       New Cairo                       Apartment, Trio                                          Apartment for sale in Trio  - with 2 bedrooms in New Cairo by M Squared. 130.0   2.0    3.0                              NaN 14,095,338 EGP    NaN
00636   

In [56]:
df_recommendations_input_json = df_recommendations_input.to_json(orient='records')
print(df_recommendations_input_json)

[{"id":"00207","location":"South New Cairo","property_name":"Apartment, DISTRICT 5","description":"Apartment for sale in DISTRICT 5 - with 2 bedrooms in South New Cairo by Marakez.","m2":149.0,"Beds":2.0,"Baths":3.0,"payment_plan":"20,000 EGP monthly \/ 5 Years","price":"13,200,000 EGP","tag":"Resale"},{"id":"00552","location":"South New Cairo","property_name":"Apartment, DISTRICT 5","description":"Apartment for sale in DISTRICT 5 - with 2 bedrooms in South New Cairo by Marakez.","m2":125.0,"Beds":2.0,"Baths":3.0,"payment_plan":"36,160 EGP monthly \/ 6 Years","price":"9,281,000 EGP","tag":"Resale"},{"id":"00556","location":"New Cairo","property_name":"Apartment, Trio","description":"Apartment for sale in Trio  - with 2 bedrooms in New Cairo by M Squared.","m2":130.0,"Beds":2.0,"Baths":3.0,"payment_plan":null,"price":"14,095,338 EGP","tag":null},{"id":"00636","location":"New Cairo","property_name":"Apartment, Address East","description":"Apartment for sale in Address East - with 2 bedro